# MapBiomas land cover: landscape portfolio for prevention planning

**Business question.** Which landscape classes dominate the Kalimantan operating area, and where should prevention programs be tailored by land-use context? This notebook uses MapBiomas Indonesia Collection 2 land-cover data only.

The raw raster class codes are retained as published. Confirm the official MapBiomas class legend before assigning sector-specific labels in external reporting.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'data').exists())
adm1 = gpd.read_file(ROOT / 'data/raw/boundaries/idn_adm1.geojson').to_crs('EPSG:4326')
name_col = next(c for c in ['shapeName', 'NAME_1', 'name'] if c in adm1.columns)
kalimantan = adm1[adm1[name_col].astype(str).str.contains('Kalimantan', case=False, na=False)]
raster_path = ROOT / 'data/raw/mapbiomas/landcover_2022_collection2.tif'
with rasterio.open(raster_path) as src:
    window = from_bounds(*kalimantan.total_bounds, transform=src.transform).round_offsets().round_lengths()
    scale = min(1.0, 1600 / max(window.width, window.height))
    height, width = max(1, round(window.height * scale)), max(1, round(window.width * scale))
    sample = src.read(1, window=window, out_shape=(height, width), resampling=Resampling.nearest, masked=True)
    sample_transform = src.window_transform(window) * Affine.scale(window.width / width, window.height / height)
inside = geometry_mask([geometry.__geo_interface__ for geometry in kalimantan.geometry], out_shape=sample.shape, transform=sample_transform, invert=True)
sample = np.ma.array(sample, mask=np.ma.getmaskarray(sample) | ~inside)

values = sample.compressed()
values = values[values != 0]
classes, counts = np.unique(values, return_counts=True)
shares = (counts / counts.sum() * 100)
landcover = {'classes': classes, 'shares': shares, 'sample': sample, 'transform': sample_transform}

In [ ]:
top = sorted(zip(landcover['classes'], landcover['shares']), key=lambda item: item[1], reverse=True)[:12]
codes, shares = zip(*top)
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
axes[0].barh([f'Class {int(code)}' for code in codes][::-1], shares[::-1], color='#4c956c')
axes[0].set(title='Largest land-cover classes in Kalimantan', xlabel='Share of sampled valid pixels (%)')
axes[1].imshow(landcover['sample'], cmap='tab20', interpolation='nearest')
axes[1].set(title='Land-cover mosaic (downsampled for EDA)', xticks=[], yticks=[])
plt.show()

top_share = sum(shares[:3])
print(f'Business insight: the three largest land-cover classes account for {top_share:.1f}% of sampled Kalimantan pixels.')
print('Recommended action: organise prevention packages by the dominant land-use contexts, then use the official MapBiomas legend to map each code to its land-management owner and intervention playbook.')

In [ ]:
# EDA tambahan: konsentrasi lanskap per provinsi untuk menentukan strategi seragam vs modular.
valid_cells = ~np.ma.getmaskarray(landcover['sample'])
rows = []
for _, province in kalimantan.iterrows():
    inside = geometry_mask([province.geometry.__geo_interface__], out_shape=landcover['sample'].shape, transform=landcover['transform'], invert=True)
    province_values = np.asarray(landcover['sample'].data[inside & valid_cells]).astype('int64')
    province_values = province_values[province_values != 0]
    province_codes, province_counts = np.unique(province_values, return_counts=True)
    top_index = province_counts.argmax()
    rows.append({'province': province[name_col], 'dominant_code': int(province_codes[top_index]), 'dominant_share_pct': province_counts[top_index] / province_counts.sum() * 100})
province_mix = pd.DataFrame(rows).sort_values('dominant_share_pct')
ax = province_mix.plot.barh(x='province', y='dominant_share_pct', legend=False, color='#2a6f97', figsize=(10, 4), title='Konsentrasi kelas tutupan lahan dominan per provinsi')
ax.set(xlabel='Pangsa kelas dominan pada sel sampel (%)', ylabel='')
for bar, code in zip(ax.patches, province_mix['dominant_code']): ax.text(bar.get_width() + .3, bar.get_y() + bar.get_height()/2, f'kode {code}', va='center')
plt.show()
print('Implikasi karhutla: provinsi dengan lanskap lebih terkonsentrasi dapat memakai paket pencegahan yang lebih fokus; provinsi yang lebih beragam membutuhkan engagement dan SOP lintas-konteks. Kode harus dipetakan ke legenda MapBiomas resmi sebelum tindakan per kelas dibuat.')

In [ ]:
# Statistik resmi Collection 4.1: perubahan komposisi provinsi 2023–2024.
temporal = pd.read_parquet(ROOT / 'data/interim/mapbiomas_province_year_features.parquet')
temporal = temporal[temporal['source_year'].isin([2023, 2024])].copy()
assert len(temporal) == 10 and temporal['province'].nunique() == 5
trend = temporal.pivot(index='province', columns='source_year', values=['lc_natural_share_pct', 'lc_anthropic_share_pct', 'lc_peat_swamp_share_pct'])
change = pd.DataFrame({
    'Natural': trend[('lc_natural_share_pct', 2024)] - trend[('lc_natural_share_pct', 2023)],
    'Antropik': trend[('lc_anthropic_share_pct', 2024)] - trend[('lc_anthropic_share_pct', 2023)],
    'Rawa gambut': trend[('lc_peat_swamp_share_pct', 2024)] - trend[('lc_peat_swamp_share_pct', 2023)],
}).sort_values('Natural')
ax = change.plot.barh(figsize=(10, 5), color=['#4c956c', '#d68c45', '#6b4c9a'], title='Perubahan pangsa tutupan per provinsi: 2024 vs 2023')
ax.axvline(0, color='black', linewidth=.8); ax.set(xlabel='Perubahan poin persentase', ylabel='')
ax.grid(axis='x', alpha=.25); plt.show()
leader = change.index[0]
print(f'Insight bisnis: {leader} mengalami penurunan pangsa natural terbesar ({change.loc[leader, "Natural"]:.2f} pp). Gunakan ini sebagai prioritas verifikasi lokasi perubahan dan engagement pengelola, bukan sebagai bukti otomatis konversi atau kenaikan risiko kebakaran.')

## Decision notes

- Raster 2022 adalah baseline lanskap, sedangkan statistik provinsi resmi tersedia untuk 2023–2024; tidak ada tren 2025–2026 yang diproyeksikan.
- The class concentration chart helps size partnerships: a few dominant landscape contexts can cover most operating area.
- Use this notebook to target land-manager engagement, canal-management checks, and outreach formats by verified class.